In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nitaisatapathy/welfake-dataset/WELFake_Dataset.csv
/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv
/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv


In [2]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f in ("Fake.csv", "True.csv") or "WELFake" in f:
            print(os.path.join(root, f))

/kaggle/input/datasets/nitaisatapathy/welfake-dataset/WELFake_Dataset.csv
/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv
/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv


In [3]:
import os, re, numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

print("GPU:", tf.config.list_physical_devices("GPU"))

# ---------- PATHS: edit these two ----------
ISOT_DIR = "/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/"
WEL_FILE = "/kaggle/input/datasets/nitaisatapathy/welfake-dataset/WELFake_Dataset.csv"

# ---------- cleaning ----------
days = r"monday|tuesday|wednesday|thursday|friday|saturday|sunday"
months = r"january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec"
places = r"washington|london|moscow|new york|beijing|brussels|berlin|paris"

def clean(t):
    t = t.lower()
    t = re.sub(r"http\S+|www\S+", " ", t)
    t = re.sub(r"pic\.twitter\.com\S*|twitter\.com\S*", " ", t)
    t = re.sub(r"\(reuters\)|reuters", " ", t)
    t = re.sub(r"21st century wire|featured image|getty images|image via|via", " ", t)
    t = re.sub(rf"\b({days}|{months}|{places})\b", " ", t)
    t = re.sub(r"\b(mr|mrs|ms)\b", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# ---------- ISOT ----------
fake = pd.read_csv(ISOT_DIR + "Fake.csv"); fake["label"] = 0
real = pd.read_csv(ISOT_DIR + "True.csv"); real["label"] = 1
df = pd.concat([fake, real])[["title", "text", "label"]].dropna().drop_duplicates(subset="text")
df["clean"] = (df["title"] + " " + df["text"]).apply(clean)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.2, stratify=df["label"], random_state=42)

isot_tr = pd.DataFrame({"text": X_train, "label": y_train}).sample(15000, random_state=42)
isot_te = pd.DataFrame({"text": X_test,  "label": y_test}).sample(3000,  random_state=42)

# ---------- WELFake (labels flipped: 0 = fake, 1 = real) ----------
w = pd.read_csv(WEL_FILE).dropna(subset=["text"])
w["title"] = w["title"].fillna("")
w["clean"] = (w["title"] + " " + w["text"]).apply(clean)
w["label"] = 1 - w["label"]
w = w.sample(20000, random_state=7)
w_tr, w_te = train_test_split(w, test_size=0.25, random_state=42, stratify=w["label"])
w_tr = w_tr[["clean", "label"]].rename(columns={"clean": "text"}).sample(10000, random_state=42)
w_te = w_te[["clean", "label"]].rename(columns={"clean": "text"}).sample(3000, random_state=42)

# ---------- models ----------
MAXLEN, VOCAB, TFIDF_DIM = 256, 30000, 10000

def build(kind):
    if kind == "ANN (TF-IDF)":
        inp = keras.Input(shape=(TFIDF_DIM,))
        x = layers.Dense(128, activation="relu")(inp)
        x = layers.Dropout(0.4)(x)
        x = layers.Dense(64, activation="relu")(x)
    else:
        inp = keras.Input(shape=(MAXLEN,))
        x = layers.Embedding(VOCAB, 100)(inp)
        if kind == "CNN":
            x = layers.Conv1D(128, 5, activation="relu")(x)
            x = layers.GlobalMaxPooling1D()(x)
        elif kind == "GRU":
            x = layers.Bidirectional(layers.GRU(64))(x)
        elif kind == "LSTM":
            x = layers.Bidirectional(layers.LSTM(64))(x)
        x = layers.Dropout(0.4)(x)
        x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model(inp, out)
    m.compile("adam", "binary_crossentropy", metrics=["accuracy"])
    return m

def run_setting(train_df, setting):
    train_df = train_df.sample(frac=1, random_state=1).reset_index(drop=True)
    ytr = train_df["label"].values

    vec = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=MAXLEN)
    vec.adapt(train_df["text"].values)
    seq = lambda s: vec(tf.constant(s.values)).numpy()
    S = (seq(train_df["text"]), seq(isot_te["text"]), seq(w_te["text"]))

    tfv = TfidfVectorizer(stop_words="english", max_features=TFIDF_DIM, ngram_range=(1, 2))
    T = (tfv.fit_transform(train_df["text"]).toarray().astype("float32"),
         tfv.transform(isot_te["text"]).toarray().astype("float32"),
         tfv.transform(w_te["text"]).toarray().astype("float32"))

    rows = []
    for kind in ["ANN (TF-IDF)", "CNN", "GRU", "LSTM"]:
        print(f"=== {setting} | {kind}")
        keras.backend.clear_session()
        Xtr, Xi, Xw = T if kind.startswith("ANN") else S
        m = build(kind)
        m.fit(Xtr, ytr, validation_split=0.1, epochs=4, batch_size=64, verbose=2,
              callbacks=[keras.callbacks.EarlyStopping(patience=1, restore_best_weights=True)])
        pi = (m.predict(Xi, verbose=0).ravel() > 0.5).astype(int)
        pw = (m.predict(Xw, verbose=0).ravel() > 0.5).astype(int)
        rows.append([setting, kind,
                     accuracy_score(isot_te["label"], pi),
                     accuracy_score(w_te["label"], pw),
                     f1_score(w_te["label"], pw)])
    return rows

rows  = run_setting(isot_tr, "A: trained on ISOT only")
rows += run_setting(pd.concat([isot_tr, w_tr], ignore_index=True), "B: trained on ISOT + WELFake")

results_dl = pd.DataFrame(rows, columns=["Setting", "Model", "ISOT acc", "WELFake acc", "WELFake F1"]).round(4)
results_dl.to_csv("/kaggle/working/results_dl.csv", index=False)
print(results_dl.to_string(index=False))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


I0000 00:00:1792569153.923128      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792569153.926905      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


=== A: trained on ISOT only | ANN (TF-IDF)
Epoch 1/4


I0000 00:00:1792569183.075879      67 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


211/211 - 6s - 26ms/step - accuracy: 0.9333 - loss: 0.1802 - val_accuracy: 0.9827 - val_loss: 0.0515
Epoch 2/4
211/211 - 1s - 4ms/step - accuracy: 0.9945 - loss: 0.0219 - val_accuracy: 0.9807 - val_loss: 0.0566
=== A: trained on ISOT only | CNN
Epoch 1/4


2026-10-21 07:53:12.411793: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=2} for conv (f32[100,128,1,5]{3,2,1,0}, u8[0]{0}) custom-call(f32[100,64,1,256]{3,2,1,0}, f32[128,64,1,252]{3,2,1,0}), window={size=1x252}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-10-21 07:53:12.426020: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.014356558s
Trying algorithm eng3{k11=2} for conv (f32[100,128,1,5]{3,2,1,0}, u8[0]{0}) custom-call(f32[100,64,1,256]{3,2,1,0}, f32[128,64,1,252]{3,2,1,0}), window={size=1x252}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", backend_config={"operation_queue_id":"0","wait_on_o

211/211 - 9s - 43ms/step - accuracy: 0.8607 - loss: 0.2943 - val_accuracy: 0.9693 - val_loss: 0.0939
Epoch 2/4
211/211 - 1s - 5ms/step - accuracy: 0.9739 - loss: 0.0733 - val_accuracy: 0.9707 - val_loss: 0.0791
Epoch 3/4
211/211 - 1s - 5ms/step - accuracy: 0.9919 - loss: 0.0259 - val_accuracy: 0.9787 - val_loss: 0.0602
Epoch 4/4
211/211 - 1s - 5ms/step - accuracy: 0.9985 - loss: 0.0068 - val_accuracy: 0.9780 - val_loss: 0.0573
=== A: trained on ISOT only | GRU
Epoch 1/4
211/211 - 9s - 44ms/step - accuracy: 0.9010 - loss: 0.1985 - val_accuracy: 0.9733 - val_loss: 0.0887
Epoch 2/4
211/211 - 5s - 21ms/step - accuracy: 0.9932 - loss: 0.0233 - val_accuracy: 0.9767 - val_loss: 0.0728
Epoch 3/4
211/211 - 4s - 21ms/step - accuracy: 0.9949 - loss: 0.0167 - val_accuracy: 0.9567 - val_loss: 0.1312
=== A: trained on ISOT only | LSTM
Epoch 1/4
211/211 - 7s - 33ms/step - accuracy: 0.9261 - loss: 0.1709 - val_accuracy: 0.9740 - val_loss: 0.0672
Epoch 2/4
211/211 - 5s - 22ms/step - accuracy: 0.9929 - 